In [13]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('read-notebook').getOrCreate()

In [14]:
# Config
spark.sparkContext.getConf().getAll()

[('spark.sql.catalog.nessie.uri', 'http://nessie:19120/iceberg'),
 ('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog'),
 ('spark.eventLog.enabled', 'true'),
 ('spark.driver.host', '485badfd96ed'),
 ('spark.hadoop.fs.s3a.path.style.access', 'true'),
 ('spark.repl.local.jars',
  'file:///opt/spark/user-jars/iceberg-spark-runtime-3.5_2.12-1.10.0.jar,file:///opt/spark/user-jars/iceberg-aws-bundle-1.10.1.jar'),
 ('spark.sql.catalog.nessie.client.region', 'us-east-1'),
 ('spark.sql.catalog.nessie.s3.endpoint', 'http://rustfs:9000'),
 ('spark.sql.catalog.nessie.prefix', 'main'),
 ('spark.app.startTime', '1773433521101'),
 ('spark.sql.catalog.nessie.s3.remote-signing-enabled', 'false'),
 ('spark.hadoop.fs.s3.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem'),
 ('spark.app.name', 'read-notebook'),
 ('spark.app.initial.jar.urls',
  'spark://485badfd96ed:33043/jars/iceberg-aws-bundle-1.10.1.jar,spark://485badfd96ed:33043/jars/iceberg-spark-runtime-3.5_2.12-1.10.0.jar'),
 ('spark.s

In [17]:
spark.sql("SHOW NAMESPACES IN nessie").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [18]:
spark.sql("SHOW TABLES IN nessie.default").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|  default|static_data|      false|
+---------+-----------+-----------+



In [19]:
spark.sql("SELECT * FROM nessie.default.static_data LIMIT 20").show()

+---+-----+--------+-----+
| id| name|category|value|
+---+-----+--------+-----+
|  1|alpha|       A| 10.5|
|  2| beta|       B| 20.0|
|  3|gamma|       A|15.75|
|  4|delta|       C|  5.0|
+---+-----+--------+-----+



In [20]:
# View all snapshots
spark.sql("SELECT * FROM nessie.default.static_data.snapshots").show(truncate=False)

+-----------------------+------------------+---------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id       |parent_id|operation|manifest_list                                                                                                                                       |summary                                                                                                   

In [21]:
# View table history
spark.sql("SELECT * FROM nessie.default.static_data.history").show(truncate=False)

+-----------------------+------------------+---------+-------------------+
|made_current_at        |snapshot_id       |parent_id|is_current_ancestor|
+-----------------------+------------------+---------+-------------------+
|2026-03-10 21:45:21.177|968374114265936035|NULL     |true               |
+-----------------------+------------------+---------+-------------------+



In [28]:
# View Manifest files
spark.sql("SELECT * FROM nessie.default.static_data.manifests").show(truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------+------+-----------------+------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
|content|path                                                                                                                         |length|partition_spec_id|added_snapshot_id |added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries|
+-------+-----------------------------------------------------------------------------------------------------------------------------+------+-----------------+------------------+----------------------+-------------------------+------------------------+------------------------+--------------

In [23]:
# View data files
spark.sql("SELECT * FROM nessie.default.static_data.files").show(truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------------+-----------+-------+------------+------------------+------------------------------------+--------------------------------+--------------------------------+----------------+--------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+-----------------------------------------------------------------------------------------------------------+
|content|file_path                                                                                                                                |file_format|spec_id|record_count|file_size_in_bytes|column_sizes                        |value_counts                    |null

In [24]:
result = spark.sql("""
    CALL nessie.system.remove_orphan_files(
        table => 'nessie.default.static_data',
        dry_run => true
    )
""")
result.show(truncate=False)

Py4JJavaError: An error occurred while calling o486.sql.
: org.apache.iceberg.exceptions.ValidationException: Cannot delete orphan files: GC is disabled (deleting files may corrupt other tables)
	at org.apache.iceberg.exceptions.ValidationException.check(ValidationException.java:49)
	at org.apache.iceberg.spark.actions.DeleteOrphanFilesSparkAction.<init>(DeleteOrphanFilesSparkAction.java:129)
	at org.apache.iceberg.spark.actions.SparkActions.deleteOrphanFiles(SparkActions.java:80)
	at org.apache.iceberg.spark.procedures.RemoveOrphanFilesProcedure.lambda$call$3(RemoveOrphanFilesProcedure.java:146)
	at org.apache.iceberg.spark.procedures.BaseProcedure.execute(BaseProcedure.java:107)
	at org.apache.iceberg.spark.procedures.BaseProcedure.withIcebergTable(BaseProcedure.java:96)
	at org.apache.iceberg.spark.procedures.RemoveOrphanFilesProcedure.call(RemoveOrphanFilesProcedure.java:143)
	at org.apache.spark.sql.execution.datasources.v2.CallExec.run(CallExec.scala:34)
	at org.apache.spark.sql.

In [26]:
# Check table properties
spark.sql("""
    SHOW TBLPROPERTIES nessie.default.static_data
""").show(truncate=False)

+-------------------------------+----------------------------------------------------------------+
|key                            |value                                                           |
+-------------------------------+----------------------------------------------------------------+
|created-at                     |2026-03-10T14:25:06.526640606Z                                  |
|current-snapshot-id            |968374114265936035                                              |
|format                         |iceberg/parquet                                                 |
|format-version                 |2                                                               |
|gc.enabled                     |false                                                           |
|nessie.catalog.content-id      |fccb06d9-60a2-40d4-9a7f-b88753c1123c                            |
|nessie.commit.id               |38c5172665dcb7d6107e6c4e1c5b5d12577f563a0d64636b6c4ab1db57c5059b|
|nessie.co

In [33]:
spark

AttributeError: 'NoneType' object has no attribute 'sc'

In [31]:
spark.stop()